# Economy of Bits — Notebook 09: eighteen years of capital

I said notebook 08 was the last. It should not have been, and the reason is my mistake
rather than a new question: **the EDGAR facts already downloaded reach back to 2007**, and
every notebook so far filtered them to 2016–2025 out of habit.

Extending the capital series changes what can be said, three ways.

**When the turn started.** On ten years the build-out looks like an AI-era event. It is not.
Microsoft's capital intensity was 0.032 in 2009 and 0.092 by 2015 — before the window opens.
Meta was at 0.243 in 2012, higher than in 2016. "AI caused this" is a claim the data can
partly refute, and it should.

**How far it has gone.** Utilities spend 0.415 of revenue on plant, real estate 0.505. The
hyperscalers reached 0.277 — two-thirds of the way from a software cost structure to a
regulated-infrastructure one, not past it.

**Intel, which is the argument in one firm.** Capital intensity 0.130 (2007) → 0.475 (2023),
outspending every hyperscaler, while its share of the top hundred fell 5.19% → 0.38%. The
firm that committed the most capital to compute captured the least value from it.

There is also something to check rather than report: Oracle comes out at 0.826 in 2025,
which is not credible, and Oracle closes its year in May. Cell 3 tests the fiscal-year rule,
because it touches every non-December filer.

Upload `bits_inputs_v9.zip`. No downloads; about two minutes.

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# I said notebook 08 was the last one. It should not have been, and the reason
# is a mistake of mine rather than a new question: the EDGAR facts already
# downloaded reach back to 2007, and every notebook so far has thrown away nine
# years of them by filtering to 2016-2025 out of habit.
#
# Extending the capital series to 2007 changes what can be said, in three ways.
#
# WHEN THE TURN STARTED. On the ten-year window the hyperscaler build-out looks
# like a recent event. It is not. Microsoft's capital intensity was 0.032 in
# 2009 and 0.092 by 2015, before the paper's window opens. Meta was at 0.243 in
# 2012, higher than in 2016. The turn is datable, and dating it properly matters
# because "AI caused this" is a claim the data can partly refute.
#
# HOW FAR IT HAS GONE. Utilities run at 0.415 of revenue in capital spending and
# real estate at 0.505. The hyperscalers reached 0.277. They are two-thirds of
# the way from a software cost structure to a regulated-infrastructure one, not
# past it, and the paper should say so in those terms rather than reaching for
# "capital-intensive" as though it were a binary.
#
# INTEL, WHICH IS THE WHOLE ARGUMENT IN ONE FIRM. Its capital intensity went
# from 0.130 in 2007 to 0.475 in 2023 -- it outspent every hyperscaler -- while
# its share of the top hundred firms fell from 5.19% to 0.38%. The firm that
# committed the most capital to compute captured the least value from it.
#
# There is also something to check rather than report. Oracle comes out at 0.826
# in 2025, which is not credible for any operating company, and Oracle closes
# its year in May. The fiscal-year assignment rule is the first suspect and cell
# 3 tests it, because the same rule touches every non-December filer.
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import stats as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

RUN_TAG = "bits_09_long_capital"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, FIGS = ROOT / "outputs", ROOT / "outputs" / "figures"
for d in (OUTD, FIGS):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet"); df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name); print(f"  saved {name}  ({len(df):,} rows)")

def trend(s, maxlags=3):
    s = s.dropna()
    if len(s) < 5:
        return np.nan, np.nan
    X = sm.add_constant(s.index.values.astype(float) - np.mean(s.index.values))
    f = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return float(f.params[1]), float(f.pvalues[1])

LONG = list(range(2008, 2026))     # 2007 has only 196 firms; 2008 has 328
print(f"writing to: {ROOT}")

## 2 · The long capital panel, 2008–2025

In [ ]:
# =============================================================================
# CELL 2 - The long capital panel, 2008-2025
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v9.zip"), recursive=True)):
        dest = ROOT / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}"); return dest
    for d in (BASE / "inputs9", BASE):
        if (pathlib.Path(d) / "edgar_facts_v3.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v9.zip not found.")

facts = pd.read_csv(SRC / "edgar_facts_v3.csv.gz")
full = pd.read_csv(SRC / "panel_full_1996.csv.gz"); full["year"] = full.year.astype(int)
rep = pd.read_csv(SRC / "panel_repaired.csv.gz"); rep["year"] = rep.year.astype(int)
t2c = pd.read_csv(SRC / "ticker_to_cik.csv")
CIK = dict(zip(t2c.ticker, t2c.cik)); CIK.update({"XOM": 34088, "BLK": 1364742})

W = facts.pivot_table(index=["cik", "year"], columns="field", values="value",
                      aggfunc="first").reset_index()
W.columns.name = None
for c in ["capex", "revenue", "ppe_net", "assets", "operating_lease_rou", "ppe_gross"]:
    if c not in W.columns:
        W[c] = np.nan
W["capex_intensity_raw"] = W.capex / W.revenue      # kept unclipped for cell 3
W["capex_intensity"] = W.capex_intensity_raw
W["asset_light"] = 1 - W.ppe_net / W.assets
W = W[W.year.isin(LONG)]
# Winsorising is applied AFTER the diagnostics in cell 3. Clipping first would
# assign every firm above the 99th percentile the same value, which hides
# exactly the mis-tagged rows the diagnostics are looking for.
for c in ["asset_light"]:
    lo = W.groupby("year")[c].transform(lambda s: s.quantile(0.01))
    hi = W.groupby("year")[c].transform(lambda s: s.quantile(0.99))
    W[c] = W[c].clip(lo, hi)

# Market caps and sectors: the repaired panel where it exists (2016-2025), the
# original before that. Stated plainly because the two are not the same object.
mc = pd.concat([
    rep[rep.year.between(2016, 2025)][["year", "ticker", "mktcap", "sector"]],
    full[full.year.between(2008, 2015)][["year", "ticker", "mktcap", "sector"]]
], ignore_index=True)
mc["cik"] = mc.ticker.map(CIK)
sec = mc.dropna(subset=["sector"]).drop_duplicates("ticker").set_index("ticker").sector
mc["sector"] = mc.sector.fillna(mc.ticker.map(sec))
L = mc.merge(W, on=["cik", "year"], how="left")
L = L.groupby(["year", "cik"], as_index=False).agg(
    ticker=("ticker", "first"), mktcap=("mktcap", "sum"), sector=("sector", "first"),
    capex_intensity=("capex_intensity", "first"),
    capex_intensity_raw=("capex_intensity_raw", "first"),
    asset_light=("asset_light", "first"),
    capex=("capex", "first"), revenue=("revenue", "first"),
    operating_lease_rou=("operating_lease_rou", "first"), assets=("assets", "first"))
L["cap_share"] = L.mktcap / L.groupby("year").mktcap.transform("sum")

print("firms with a capital-intensity figure, by year:")
print(L[L.capex_intensity.notna()].groupby("year").cik.nunique().to_string())
print("\n2016-2025 uses the REPAIRED panel; 2008-2015 uses the original, which")
print("has 69-79% roster coverage and no rename repair. Anything said about the")
print("early years is said about the firms that survived to be in it.")
savetable(L, "long_capital_panel")

## 3 · Check the fiscal-year rule before trusting anything built on it

In [ ]:
# =============================================================================
# CELL 3 - Check the inputs before trusting anything built on them
#
# Two problems, found by looking at the extreme values rather than assuming they
# were real.
#
# ONE, REVENUE IS MIS-TAGGED FOR SOME FILERS. Charter comes out with revenue of
# $1.1bn against capital spending of $9.4bn; its actual revenue is around $55bn.
# American Tower, Crown Castle, Extra Space and Healthpeak show the same
# pattern. The `Revenues` tag picks up a component rather than the total for
# these filers, and it inflates their capital intensity without limit -- which
# in turn inflates the Real Estate and Communication Services sector averages
# that cell 5 uses as benchmarks.
#
# TWO, FISCAL YEARS. Oracle appears at 0.826 in 2025 and closes its year in May.
# The rule assigns a period to the calendar year containing its midpoint, so the
# row labelled 2025 for a May filer is the year ending May 2026, which can be
# incomplete.
#
# Both are caught the same way: a firm's revenue should not collapse against its
# own history, and capital spending should not exceed revenue for a going
# concern. Rows failing either test are set to missing, not quietly clipped.
# =============================================================================
L["rev_median"] = L.groupby("cik").revenue.transform("median")
L["rev_vs_own_median"] = L.revenue / L.rev_median
L["ci"] = L.capex_intensity_raw

BAD_REV = (L.rev_vs_own_median < 0.5) & L.rev_vs_own_median.notna()
BAD_RATIO = (L.ci > 1.0) & L.ci.notna()
BAD = BAD_REV | BAD_RATIO
print(f"{int(BAD_REV.sum())} firm-years where revenue is below half the firm's own median")
print(f"{int(BAD_RATIO.sum())} firm-years where capital spending exceeds revenue")
print(f"{int(BAD.sum())} excluded in total, out of {len(L):,}")

worst = L[BAD].sort_values("ci", ascending=False)
print("\nthe twenty worst, with the raw inputs so the problem is visible:")
display(worst.head(20)[["year", "ticker", "sector", "capex", "revenue",
                        "rev_median", "ci"]].round(3))
print("\nCharter's revenue is about $55bn; American Tower's about $11bn. Where")
print("the revenue column shows a fraction of that, the tag is wrong.")

print("\nfirms most often excluded:")
print(L[BAD].ticker.value_counts().head(15).to_string())
print("\nnon-December filers among the exclusions:")
NONDEC = {"ORCL": "May", "MSFT": "June", "AAPL": "September", "NVDA": "January",
          "WMT": "January", "COST": "August", "AVGO": "October", "ADBE": "November",
          "CSCO": "July", "QCOM": "September", "DE": "October", "TGT": "January"}
nd = sorted(set(L[BAD].ticker) & set(NONDEC))
print(f"  {nd if nd else 'none'}")

print("\nOracle year by year, before exclusion:")
display(L[L.ticker == "ORCL"].sort_values("year")
        [["year", "capex", "revenue", "rev_median", "ci"]].round(3))

savetable(L[BAD][["year", "ticker", "sector", "capex", "revenue", "rev_median", "ci"]],
          "excluded_firm_years")
L.loc[BAD, ["capex_intensity", "capex_intensity_raw", "capex", "revenue"]] = np.nan

# Winsorise only now, on data that has already had its errors removed.
for c in ["capex_intensity", "asset_light"]:
    lo = L.groupby("year")[c].transform(lambda s: s.quantile(0.01))
    hi = L.groupby("year")[c].transform(lambda s: s.quantile(0.99))
    L[c] = L[c].clip(lo, hi)
print(f"\n{int(L.capex_intensity.notna().sum()):,} firm-years survive with a "
      f"capital-intensity figure")
print("\nsector means after cleaning -- compare with the pre-clean figures that")
print("put Real Estate at 0.505, which was Charter and the tower REITs:")
display(L.pivot_table(index="year", columns="sector", values="capex_intensity")
        .loc[[2016, 2020, 2025]].round(3).T.sort_values(2025, ascending=False))

## 4 · When did the build-out actually start?

In [ ]:
# =============================================================================
# CELL 4 - When did the build-out actually start?
#
# The ten-year window makes the hyperscaler turn look like an AI-era event. With
# 2008 onward in view it can be dated, and the date is the point: a trend that
# began around 2013 is a cloud story that AI accelerated, not an AI story.
# =============================================================================
HYPER = {"MSFT", "GOOGL", "GOOG", "AMZN", "META"}
DESIGN = {"NVDA", "AVGO", "QCOM", "AMD", "SNPS", "CDNS", "ANET", "MRVL", "XLNX", "ALTR"}
FAB = {"INTC", "TXN", "MU", "ADI", "MCHP", "ON", "NXPI", "SWKS", "QRVO"}
def grp(t):
    t = str(t).upper()
    return ("hyperscaler" if t in HYPER else "designs" if t in DESIGN
            else "fabricates" if t in FAB else "rest of index")
L["grp"] = L.ticker.map(grp)

piv = L.pivot_table(index="year", columns="grp", values="capex_intensity")
print("capital intensity by group, 2008-2025:")
display(piv.round(3))
savetable(piv.reset_index(), "capex_by_group_long")

print("\ntrends estimated over three windows:")
rows = []
for g in piv.columns:
    for lo, hi, lab in [(2008, 2015, "2008-2015, before the paper's window"),
                        (2016, 2025, "2016-2025, the paper's window"),
                        (2008, 2025, "2008-2025, all")]:
        s = piv[g][(piv.index >= lo) & (piv.index <= hi)]
        b, p = trend(s)
        if b == b:
            rows.append({"group": g, "window": lab,
                         "first": round(float(s.dropna().iloc[0]), 3),
                         "last": round(float(s.dropna().iloc[-1]), 3),
                         "slope": round(b, 5), "p": round(p, 4)})
wt = pd.DataFrame(rows)
display(wt)
savetable(wt, "capex_trends_three_windows")

print("\nfirm by firm, the four hyperscalers plus Intel:")
show = L[L.ticker.isin(["MSFT", "GOOGL", "GOOG", "AMZN", "META", "INTC"])]
display(show.pivot_table(index="year", columns="ticker", values="capex_intensity").round(3))

print("\nWhen does each hyperscaler first exceed the rest of the index?")
restline = piv["rest of index"]
for t in ["MSFT", "AMZN", "GOOGL", "META"]:
    s = L[L.ticker.isin([t, "GOOG"] if t == "GOOGL" else [t])] \
        .groupby("year").capex_intensity.mean().dropna()
    above = [y for y in s.index if y in restline.index and s[y] > restline[y]]
    print(f"  {t:<6} first year above the index average: "
          f"{min(above) if above else 'never'};  "
          f"continuously above from {min(y for y in above if all(x in above for x in range(y, 2026))) if above else '-'}")
print("\nIf the hyperscalers were already above the index average years before")
print("2016, the paper cannot present the capital turn as an AI phenomenon. It")
print("is a cloud-era reallocation with an AI-era acceleration on top, and those")
print("are different claims with different implications for what comes next.")

## 5 · How far along the spectrum, and the Intel counter-case

In [ ]:
# =============================================================================
# CELL 5 - How far along the spectrum, and the Intel counter-case
#
# "Capital-intensive" is not a binary and the paper should not use it as one.
# Utilities spend 0.415 of revenue on plant and real estate 0.505. The
# hyperscalers reached 0.277. Placing them on that spectrum is more informative
# than any adjective, and it is the honest way to report a level.
# =============================================================================
bench = L.pivot_table(index="year", columns="sector", values="capex_intensity")
hyper = L[L.grp == "hyperscaler"].groupby("year").capex_intensity.mean()
cmpx = pd.DataFrame({"hyperscalers": hyper})
for s in ["Utilities", "Real Estate", "Energy", "Communication Services",
          "Information Technology", "Industrials"]:
    if s in bench.columns:
        cmpx[s] = bench[s]
print("capital intensity, hyperscalers against genuinely capital-heavy sectors:")
display(cmpx.round(3))
savetable(cmpx.reset_index(), "capital_intensity_benchmark")

for y in [2008, 2015, 2020, 2025]:
    if y in cmpx.index and "Utilities" in cmpx.columns:
        h, u, it = cmpx.hyperscalers.get(y), cmpx.Utilities.get(y), \
                   cmpx.get("Information Technology", pd.Series()).get(y)
        if pd.notna(h) and pd.notna(u) and pd.notna(it):
            print(f"  {y}: hyperscalers {h:.3f}  |  IT sector {it:.3f}  |  "
                  f"utilities {u:.3f}  |  {100*(h-it)/(u-it):.0f}% of the way "
                  f"from software to utility")

print("\n" + "=" * 72)
print("INTEL: THE FIRM THAT SPENT THE MOST AND CAPTURED THE LEAST")
print("=" * 72)
intel = L[L.ticker == "INTC"].sort_values("year")
intel_share = 100 * intel.set_index("year").cap_share
tbl = pd.DataFrame({"capex_intensity": intel.set_index("year").capex_intensity,
                    "share_of_index_pct": intel_share}).round(3)
display(tbl)
b1, p1 = trend(tbl.capex_intensity); b2, p2 = trend(tbl.share_of_index_pct)
print(f"  capital intensity   slope {b1:+.5f}/yr, p = {p1:.4f}")
print(f"  share of index      slope {b2:+.5f}/yr, p = {p2:.4f}")
r = tbl.corr().iloc[0, 1]
print(f"  correlation between the two: {r:+.3f}")
savetable(tbl.reset_index(), "intel_case")

nv = L[L.ticker == "NVDA"].sort_values("year")
if len(nv):
    print("\nNvidia over the same years, for contrast:")
    display(pd.DataFrame({"capex_intensity": nv.set_index("year").capex_intensity,
                          "share_of_index_pct": 100 * nv.set_index("year").cap_share}).round(3))
print("\nTwo firms in the same industry, selling to the same customers. One")
print("built the plant; the other bought capacity from TSMC. The paper does not")
print("need to argue that owning the fab CAUSED the divergence -- Intel's")
print("problems are also execution and process failures, and that belongs in the")
print("text. What the data shows is that the capital commitment and the value")
print("capture went in opposite directions, which is the fact the bit-economics")
print("account does not predict.")

## 6 · Leases, settled with the actual number

In [ ]:
# =============================================================================
# CELL 6 - Leases, settled with the actual number
#
# Notebook 05 answered the ASC 842 objection three ways without ever seeing the
# lease figure itself. It is in the data now: OperatingLeaseRightOfUseAsset,
# reported from 2016. Subtracting it from total assets gives asset lightness on
# a pre-lease basis, and the difference between the two series is the exact size
# of the accounting effect rather than an argument about it.
# =============================================================================
Lz = L.copy()
Lz["rou_share"] = Lz.operating_lease_rou / Lz.assets
Lz["asset_light_ex_lease"] = 1 - (Lz.assets * (1 - Lz.asset_light)) / \
                             (Lz.assets - Lz.operating_lease_rou.fillna(0))
have = Lz[Lz.operating_lease_rou.notna()]
print(f"{len(have):,} firm-years report an operating-lease right-of-use asset, "
      f"{have.year.min()}-{have.year.max()}")
print("\nlease assets as a share of total assets, by group:")
display(Lz.pivot_table(index="year", columns="grp", values="rou_share").round(4))

print("\nasset lightness for the hyperscalers, as reported and excluding leases:")
h = Lz[Lz.grp == "hyperscaler"].groupby("year")[
    ["asset_light", "asset_light_ex_lease", "rou_share"]].mean()
display(h.round(4))
for c in ["asset_light", "asset_light_ex_lease"]:
    b, p = trend(h[c])
    print(f"  {c:<22} {h[c].dropna().iloc[0]:.4f} -> {h[c].dropna().iloc[-1]:.4f}   "
          f"slope {b:+.5f}, p = {p:.4f}")
print("\nIf the two slopes are close, the lease question is closed with a number")
print("instead of an argument, and the paper can say so in one sentence.")
savetable(h.reset_index(), "lease_adjusted_asset_lightness")

## 7 · Figures and the updated fact sheet

In [ ]:
# =============================================================================
# CELL 7 - Figures, and the updated fact sheet
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, ACC, GREY, GRN = "#1f3b57", "#c8702a", "#7a7a7a", "#4a7c59"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
ax = axes[0]
for g, c, ls in [("hyperscaler", INK, "-"), ("designs", ACC, "-"),
                 ("fabricates", GRN, "--"), ("rest of index", GREY, ":")]:
    if g in piv.columns:
        s = piv[g].dropna()
        ax.plot(s.index, s.values, ls, color=c, lw=2, label=g)
ax.axvspan(2016, 2025, color=INK, alpha=0.06, lw=0)
ax.annotate("the paper's window", (2020.5, ax.get_ylim()[1] * 0.93), ha="center",
            fontsize=7, color=GREY)
ax.set_xlabel("year"); ax.set_ylabel("capex / revenue")
ax.set_title("Dating the build-out"); ax.legend(fontsize=8)

ax = axes[1]
for c, col, ls in [("hyperscalers", INK, "-"), ("Utilities", GRN, "--"),
                   ("Real Estate", GREY, ":"), ("Information Technology", ACC, "-")]:
    if c in cmpx.columns:
        s = cmpx[c].dropna()
        ax.plot(s.index, s.values, ls, color=col, lw=2, label=c)
ax.set_xlabel("year"); ax.set_ylabel("capex / revenue")
ax.set_title("How far along the spectrum"); ax.legend(fontsize=7.5)

ax = axes[2]
ax2 = ax.twinx()
ax.plot(tbl.index, tbl.capex_intensity, "o-", color=ACC, lw=2, ms=4)
ax2.plot(tbl.index, tbl.share_of_index_pct, "s-", color=INK, lw=2, ms=4)
ax.set_ylabel("capex / revenue", color=ACC); ax.tick_params(axis="y", labelcolor=ACC)
ax2.set_ylabel("% of index value", color=INK); ax2.tick_params(axis="y", labelcolor=INK)
ax.set_xlabel("year"); ax.set_title("Intel: spending up, value down")
fig.suptitle("Notebook 09: eighteen years of capital", fontsize=10, color=GREY, y=1.02)
savefig(fig, "L1_long_capital")

facts = pd.read_csv(SRC / "paper_facts.csv")
new = []
def add(sec, q, v):
    new.append({"section": sec, "quantity": q, "value": str(v), "source": "nb09"})
for g in ["hyperscaler", "designs", "fabricates", "rest of index"]:
    r = wt[(wt.group == g) & (wt.window.str.startswith("2008-2015"))]
    if len(r):
        add("dating", f"{g} capex slope 2008-2015",
            f"{r.slope.iloc[0]:+.5f}/yr, p={r.p.iloc[0]}")
if "Utilities" in cmpx.columns:
    add("calibration", "hyperscaler vs utilities capex 2025",
        f"{cmpx.hyperscalers.get(2025):.3f} vs {cmpx.Utilities.get(2025):.3f}")
add("counter-case", "Intel capex intensity 2008 -> 2025",
    f"{tbl.capex_intensity.dropna().iloc[0]:.3f} -> {tbl.capex_intensity.dropna().iloc[-1]:.3f}")
add("counter-case", "Intel share of index 2008 -> 2025",
    f"{tbl.share_of_index_pct.dropna().iloc[0]:.2f}% -> {tbl.share_of_index_pct.dropna().iloc[-1]:.2f}%")
add("leases", "hyperscaler asset lightness, reported vs ex-lease slope",
    f"see lease_adjusted_asset_lightness")
allfacts = pd.concat([facts, pd.DataFrame(new)], ignore_index=True)
display(allfacts)
savetable(allfacts, "paper_facts_v2")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
pd.DataFrame([{"file": str(p.relative_to(OUTD)), "kb": round(p.stat().st_size/1024, 1)}
              for p in files]).to_csv(OUTD / "MANIFEST.csv", index=False)
(OUTD / "run_summary.json").write_text(json.dumps(
    {"run": RUN_TAG, "when": datetime.datetime.now().isoformat(timespec="seconds"),
     "tables": saved_tables}, indent=2, default=str))

print("\n" + "=" * 70)
print("SEND ME")
print("=" * 70)
print("  capex_trends_three_windows.csv   when the build-out started")
print("  capital_intensity_benchmark.csv  how far along the spectrum")
print("  intel_case.csv                   the counter-case")
print("  lease_adjusted_asset_lightness.csv")
print("  part_period_exclusions.csv       whether the Oracle figure was real")
print("  paper_facts_v2.csv               everything, in one file")
print("\nThis one really is the last. The next thing I send is a draft.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive); print("Download started.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")